# 🧠 의도 파악 분류 모델 학습 (nlu-intent-v1, Colab)

백엔드 규칙이 못 알아들은 손님 말을 **키오스크 표준 문장 18개 중 하나**(또는 모름)로 바꾸는 작은 한국어 모델을 학습합니다.
예: (식사 장소 단계) "봉투에 담아 줘" → `포장`, (쿠폰 단계) "그런 거 없는데" → `없어요`

- 기반 모델: `klue/roberta-small` (한국어 BERT 계열, 6800만 개 파라미터)
- 결과: `nlu-intent-v1.zip` (ONNX 모델 + 토크나이저 + 설정). 모델 서버 `/nlu` 가 CPU로 0.1초 안에 답합니다.

| 단계 | 하는 일 | 걸리는 시간 (A100) |
|---|---|---|
| 0 | GPU 확인 · 라이브러리 설치 | 1~2분 |
| 1 | 데이터 올리기 (`nlu_data.zip`) | 1분 |
| 2 | 데이터 확인 | 바로 |
| 3 | 학습 | 3~5분 |
| 4 | 사람이 쓴 **처음 보는 말** 시험 · 기준값 정하기 | 바로 |
| 5 | ONNX 변환 · 작게 줄이기(int8) · 속도 확인 | 1~2분 |
| 6 | 결과 zip 내려받기 (+ Drive 저장) | 바로 |

**사용법:** 메뉴 `런타임 → 런타임 유형 변경 → A100 GPU` 선택 후, 위에서부터 셀을 하나씩 실행(▶)하세요. 1단계에서 파일 선택 창이 뜨면 `nlu_data.zip` 을 고릅니다.

## 0. GPU 확인 · 라이브러리 설치

In [ ]:
import subprocess, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (CPU로 느리게 돌아갑니다)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 1. 데이터 올리기
`nlu_data.zip` 을 올립니다. Google Drive 의 `MyDrive/fundamental1/nlu_data.zip` 에 넣어 두었으면 그걸 씁니다.

In [ ]:
import os, zipfile, shutil
WORK = "/content/nlu" if IN_COLAB else os.environ.get("NLU_WORK", "./nlu_work")
os.makedirs(WORK, exist_ok=True)
ZIP = os.path.join(WORK, "nlu_data.zip")
DRIVE_ZIP = "/content/drive/MyDrive/fundamental1/nlu_data.zip"

if not os.path.exists(ZIP):
    if IN_COLAB:
        from google.colab import drive, files
        try:
            drive.mount("/content/drive")
        except Exception as e:
            print("Drive 연결 안 함:", e)
        if os.path.exists(DRIVE_ZIP):
            shutil.copy(DRIVE_ZIP, ZIP)
            print("Drive 에서 가져옴")
        else:
            print("nlu_data.zip 을 골라 주세요")
            up = files.upload()
            name = next(iter(up))
            shutil.move(name, ZIP)
    else:
        shutil.copy(os.environ["NLU_DATA_ZIP"], ZIP)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(WORK)
DATA = os.path.join(WORK, "nlu")
print(sorted(os.listdir(DATA)))

## 2. 데이터 확인

In [ ]:
import csv, json, random, collections
def read_csv(name):
    with open(os.path.join(DATA, name), encoding="utf-8") as f:
        return list(csv.DictReader(f))
train, val, test = read_csv("cls_train.csv"), read_csv("cls_val.csv"), read_csv("testset.csv")
meta = json.load(open(os.path.join(DATA, "meta.json"), encoding="utf-8"))
LABELS = meta["labels"]; NONE = meta["none_label"]; STEP_NAMES = meta["step_names"]
L2I = {l: i for i, l in enumerate(LABELS)}

def model_input(step, text):
    # 서버(model/app/nlu_intent.py)도 meta.json 의 input_format 으로 똑같이 만든다
    return meta["input_format"].format(step=STEP_NAMES.get(step, step or ""), text=text)

print(f"학습 {len(train)} · 검증 {len(val)} · 시험(처음 보는 말) {len(test)} · 라벨 {len(LABELS)}개")
print(collections.Counter(r["label"] for r in train).most_common())
for r in random.Random(0).sample(train, 8):
    print(" ", model_input(r["step"], r["text"]), "→", r["label"])

## 3. 학습

In [ ]:
import time, numpy as np
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

BASE = "klue/roberta-small"
MAX_LEN = 64
EPOCHS = int(os.environ.get("NLU_EPOCHS", 8))
BATCH = 32
LR = 5e-5
torch.manual_seed(0)

tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=L2I).to(DEVICE)

def batches(rows, shuffle):
    def collate(b):
        enc = tok([model_input(r["step"], r["text"]) for r in b], padding=True, truncation=True,
                  max_length=MAX_LEN, return_tensors="pt", return_token_type_ids=False)
        enc["labels"] = torch.tensor([L2I[r["label"]] for r in b])
        return enc
    return DataLoader(rows, batch_size=BATCH, shuffle=shuffle, collate_fn=collate)

@torch.no_grad()
def predict(rows):
    model.eval()
    probs = []
    for b in batches(rows, False):
        b = {k: v.to(DEVICE) for k, v in b.items() if k != "labels"}
        probs.append(torch.softmax(model(**b).logits.float(), -1).cpu().numpy())
    return np.concatenate(probs)

def accuracy(rows):
    p = predict(rows)
    return float(np.mean([LABELS[i] == r["label"] for i, r in zip(p.argmax(-1), rows)]))

loader = batches(train, True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total = EPOCHS * len(loader)
sched = get_linear_schedule_with_warmup(opt, int(total * 0.1), total)
best, best_state = -1, None
t0 = time.time()
for epoch in range(EPOCHS):
    model.train()
    for b in loader:
        b = {k: v.to(DEVICE) for k, v in b.items()}
        loss = model(**b).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad()
    va, te = accuracy(val), accuracy(test)
    print(f"epoch {epoch + 1}/{EPOCHS} · loss {loss.item():.3f} · 검증 {va:.1%} · 처음 보는 말 {te:.1%} · {time.time() - t0:.0f}초")
    if va > best:  # 시험 데이터로 고르지 않는다 (검증으로만 고름)
        best = va
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
print(f"가장 좋은 검증 정확도: {best:.1%}")

## 4. 처음 보는 말 시험 · 기준값(threshold) 정하기
모델이 확신(확률)이 기준값보다 낮으면 "모름"으로 답합니다 → 키오스크는 "다시 말씀해 주세요".
**틀린 표준 문장으로 바꾸는 것(오답)** 이 "모름"보다 나쁘므로, 오답이 적은 기준값을 고릅니다.

In [ ]:
probs = predict(test)
def score(th):
    right = wrong = unsure = 0
    for p, r in zip(probs, test):
        i = int(p.argmax()); label = LABELS[i]
        answer = label if (label != NONE and p[i] >= th) else NONE
        if answer == r["label"]:
            right += 1
        elif answer == NONE:
            unsure += 1   # 모른다고 함 (다시 묻기)
        else:
            wrong += 1    # 다른 뜻으로 바꿈 (나쁨)
    return right, wrong, unsure
print("기준값 · 맞음 · 오답 · 모름")
for th in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    r, w, u = score(th)
    print(f"  {th:.2f} · {r / len(test):.1%} · {w / len(test):.1%} · {u / len(test):.1%}")

# 오답 2% 이하인 것 중 가장 많이 맞히는 기준값
cands = [(score(th), th) for th in np.arange(0.5, 0.96, 0.05)]
ok = [(s[0], th) for s, th in cands if s[1] / len(test) <= 0.02] or [(cands[-1][0][0], cands[-1][1])]
THRESHOLD = float(round(max(ok)[1], 2))
r, w, u = score(THRESHOLD)
print(f"\n→ 기준값 {THRESHOLD}: 맞음 {r / len(test):.1%} · 오답 {w / len(test):.1%} · 모름 {u / len(test):.1%}")
print("\n틀리거나 모른 문장:")
for p, row in zip(probs, test):
    i = int(p.argmax()); label = LABELS[i]
    answer = label if (label != NONE and p[i] >= THRESHOLD) else NONE
    if answer != row["label"]:
        print(f"  [{row['step']}] {row['text']} → {answer} ({p[i]:.2f}) / 정답 {row['label']}")

## 5. ONNX 변환 · int8 로 줄이기 · CPU 속도 확인

In [ ]:
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

OUT = os.path.join(WORK, "nlu-intent-v1")
os.makedirs(OUT, exist_ok=True)
model.eval().to("cpu")
dummy = tok([model_input("MENU", "포장해 주세요")], return_tensors="pt", return_token_type_ids=False)
fp32 = os.path.join(WORK, "model_fp32.onnx")
args = dict(input_names=["input_ids", "attention_mask"], output_names=["logits"],
            dynamic_axes={"input_ids": {0: "b", 1: "t"}, "attention_mask": {0: "b", 1: "t"}, "logits": {0: "b"}},
            opset_version=17)
try:
    torch.onnx.export(model, (dummy["input_ids"], dummy["attention_mask"]), fp32, dynamo=False, **args)
except TypeError:  # 예전 torch
    torch.onnx.export(model, (dummy["input_ids"], dummy["attention_mask"]), fp32, **args)
int8 = os.path.join(OUT, "model.onnx")
quantize_dynamic(fp32, int8, weight_type=QuantType.QInt8)

def onnx_probs(path, rows):
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    out = []
    for r in rows:
        e = tok([model_input(r["step"], r["text"])], truncation=True, max_length=MAX_LEN, return_tensors="np",
                return_token_type_ids=False)
        logits = sess.run(None, {"input_ids": e["input_ids"].astype(np.int64),
                                 "attention_mask": e["attention_mask"].astype(np.int64)})[0][0]
        x = np.exp(logits - logits.max()); out.append(x / x.sum())
    return np.array(out)

p32, p8 = onnx_probs(fp32, test), onnx_probs(int8, test)
acc = lambda p: np.mean([LABELS[i] == r["label"] for i, r in zip(p.argmax(-1), test)])
print(f"처음 보는 말 정확도(기준값 없이): PyTorch {acc(probs):.1%} · ONNX {acc(p32):.1%} · int8 {acc(p8):.1%}")
if acc(p8) < acc(p32) - 0.02:
    print("int8 이 2%p 넘게 떨어져서 원래 크기 모델을 씁니다")
    shutil.copy(fp32, int8)
    probs_final = p32
else:
    probs_final = p8
print(f"모델 크기: {os.path.getsize(int8) / 1e6:.0f}MB")

sess = ort.InferenceSession(int8, providers=["CPUExecutionProvider"])
e = tok([model_input("DINE_TYPE", "봉투에 담아 줘")], return_tensors="np", return_token_type_ids=False)
feed = {"input_ids": e["input_ids"].astype(np.int64), "attention_mask": e["attention_mask"].astype(np.int64)}
t0 = time.time()
for _ in range(50):
    sess.run(None, feed)
print(f"CPU 한 문장: {(time.time() - t0) / 50 * 1000:.1f}ms")

## 6. 결과 저장 · 내려받기

In [ ]:
tok.backend_tokenizer.save(os.path.join(OUT, "tokenizer.json"))
r, w, u = score(THRESHOLD)
info = {
    **meta, "model_version": "nlu-intent-v1", "base_model": BASE, "max_len": MAX_LEN, "threshold": THRESHOLD,
    "metrics": {"val_acc": round(best, 4), "test_right": round(r / len(test), 4), "test_wrong": round(w / len(test), 4),
                "test_unsure": round(u / len(test), 4), "test_size": len(test)},
    "trained_at": time.strftime("%Y-%m-%d %H:%M"),
}
json.dump(info, open(os.path.join(OUT, "meta.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(json.dumps(info["metrics"], ensure_ascii=False))

result = shutil.make_archive(os.path.join(WORK, "nlu-intent-v1"), "zip", WORK, "nlu-intent-v1")
print("결과:", result, f"{os.path.getsize(result) / 1e6:.0f}MB")
if IN_COLAB:
    if os.path.isdir("/content/drive/MyDrive"):
        os.makedirs("/content/drive/MyDrive/fundamental1", exist_ok=True)
        shutil.copy(result, "/content/drive/MyDrive/fundamental1/nlu-intent-v1.zip")
        print("Drive 에도 저장: MyDrive/fundamental1/nlu-intent-v1.zip")
    from google.colab import files
    files.download(result)

## 다 했으면
내려받은 `nlu-intent-v1.zip` 을 `C:\Users\USER\Documents\프로젝트1\` 에 두고 Claude 에게 "학습 끝났어" 라고 알려 주세요.
Claude 가 GitHub Release 에 올리고 모델 서버에 넣어 시험합니다. 위 4단계 숫자(맞음·오답·모름)는 노션 "모델 실험 기록"에 남깁니다.